# qfa_ocftosales 因子

qfa_ocftosales：单季度经营性现金流量净额 / 营业收入

## 裸因子指标计算

In [2]:
# -*- coding: utf-8 -*-
"""
BigQuant / DAI 复现华泰财务质量因子 qfa_ocftosales：裸因子指标测试批量低内存版 v5。

适用场景：
- v4 安全版可以跑通，但每个截面要单独查 3 次 DAI，速度太慢；
- 本版把多个 signal_date 合并成一个 chunk，每个 chunk 只做 3 次 DAI 查询：
    1) signal_date 的因子、收盘价、市值、ST/停牌状态；
    2) next_date 的下一交易日停牌状态；
    3) fwd_date 的未来收盘价与沪深300收盘价；
  然后在 Python 侧按日期映射 merge 并计算指标。

因子口径：
    qfa_ocftosales = 单季度经营性现金流量净额 / 单季度营业收入
默认 BigQuant 口径：
    cn_stock_prefactors.net_cffoa_mrq / NULLIF(cn_stock_prefactors.operating_revenue_mrq, 0)

严谨性：
- 信号只使用 signal_date 当日已存在的因子、价格、市值、风险警示、停牌和北交所状态；
- 下一交易日停牌只用于过滤不可执行股票，和你之前指标测试代码口径保持一致；
- 未来 forward_days 收益只用于事后评价，不参与选股；
- 裸因子不做行业/市值中性化，只做截面 MAD 去极值与标准化；
- 回归法使用未来 forward_days 日相对沪深300超额收益 ~ const + 标准化因子，WLS权重为 sqrt(流通市值)。

速度优化：
- 避免逐截面 3 次查询；
- 避免 SQL 自连接和窗口函数；
- 每个 chunk 处理完成后立即释放内存；
- 默认不画图，避免字体/渲染开销。
"""

import gc
import os
import re
import time
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except ImportError:
    from bigquant import dai  # type: ignore

warnings.filterwarnings("ignore")

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


@dataclass(frozen=True)
class FactorSource:
    table: str
    expr: str
    desc: str


@dataclass(frozen=True)
class Config:
    start_date: str = "2023-01-01"
    end_date: str = "2026-06-30"

    forward_days: int = 30
    rebalance_freq: int = 30
    min_cross_section_size: int = 30

    factor_name: str = "qfa_ocftosales_raw"

    # 批量大小：每个 chunk 包含多少个 signal_date。
    # 如果内存吃紧改为 4；如果运行稳定可改为 12 或 16。
    chunk_signal_dates: int = 8
    max_signal_dates: Optional[int] = None  # 调试时可设为 3；正式跑设为 None

    make_plots: bool = False
    show_progress: bool = True
    save_metrics_csv: bool = False
    metrics_csv_path: str = "qfa_ocftosales_metrics_checkpoint.csv"

    # 防止收入接近 0 导致极端比值冲击数值稳定性。
    factor_abs_cap: float = 1e6

    auto_detect_factor_source: bool = False
    factor_sources: Tuple[FactorSource, ...] = field(default_factory=lambda: (
        FactorSource(
            "cn_stock_prefactors",
            "net_cffoa_mrq / NULLIF(operating_revenue_mrq, 0)",
            "单季度经营活动现金流净额 / 单季度营业收入，默认推荐口径",
        ),
        FactorSource("cn_stock_prefactors", "qfa_ocftosales", "可能的 Wind 同名字段"),
        FactorSource("cn_stock_prefactors", "ocftosales_mrq", "可能的单季度现金流收入比字段"),
        FactorSource("cn_stock_prefactors", "cffoa_to_revenue_mrq", "可能的单季度现金流收入比字段"),
        FactorSource("cn_stock_factors_financial_indicators", "qfa_ocftosales", "可能的 Wind 同名字段"),
        FactorSource("cn_stock_factors_financial_indicators", "ocftosales_mrq", "可能的单季度现金流收入比字段"),
        FactorSource("cn_stock_factors_financial_indicators", "cffoa_to_revenue_mrq", "可能的单季度现金流收入比字段"),
        FactorSource(
            "cn_stock_factors_financial_indicators",
            "net_cffoa_mrq / NULLIF(operating_revenue_mrq, 0)",
            "单季度经营活动现金流净额 / 单季度营业收入，备用计算口径",
        ),
    ))


CFG = Config()
_T0 = time.time()


# =========================
# 基础工具
# =========================

def _elapsed() -> str:
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg: str, cfg: Config = CFG) -> None:
    if cfg.show_progress:
        print(f"[{_elapsed()}] {msg}", flush=True)


def query_df(sql: str) -> pd.DataFrame:
    return dai.query(sql).df()


def mem_mb() -> float:
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1024 / 1024
    except Exception:
        return np.nan


def date_in_sql(dates) -> str:
    return ", ".join(f"DATE '{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates)


def _short_expr(expr: str, max_len: int = 100) -> str:
    expr = " ".join(str(expr).split())
    return expr if len(expr) <= max_len else expr[: max_len - 3] + "..."


_SQL_KEYWORDS = {
    "NULLIF", "COALESCE", "ABS", "CASE", "WHEN", "THEN", "ELSE", "END", "CAST", "AS",
    "DATE", "TRUE", "FALSE", "AND", "OR", "NOT", "IS", "NULL", "IN", "BETWEEN",
    "IF", "GREATEST", "LEAST", "POWER", "SQRT", "LOG", "LN", "EXP",
}


def qualify_expr(expr: str, alias: str) -> str:
    """将表达式里的字段名加表别名，避免 join 后字段歧义。"""
    def repl(match: re.Match) -> str:
        token = match.group(0)
        if token.upper() in _SQL_KEYWORDS:
            return token
        return f"{alias}.{token}"
    return re.sub(r"(?<!\.)\b[A-Za-z_][A-Za-z0-9_]*\b", repl, expr)


def test_factor_sources(cfg: Config = CFG) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for src in cfg.factor_sources:
        expr = qualify_expr(src.expr, "f")
        sql = f"""
        SELECT f.date, f.instrument, ({expr}) AS factor_raw
        FROM {src.table} AS f
        WHERE f.date >= DATE '{cfg.start_date}'
          AND f.date <= DATE '{cfg.end_date}'
          AND ({expr}) IS NOT NULL
          AND ABS(({expr})) < {cfg.factor_abs_cap}
        LIMIT 5
        """
        try:
            tmp = query_df(sql)
            ok = tmp is not None and not tmp.empty
            rows.append({
                "table": src.table,
                "expr": _short_expr(src.expr),
                "desc": src.desc,
                "available": ok,
                "rows": 0 if tmp is None else len(tmp),
                "error": "" if ok else "查询成功但无非空样本",
            })
        except Exception as e:
            rows.append({
                "table": src.table,
                "expr": _short_expr(src.expr),
                "desc": src.desc,
                "available": False,
                "rows": 0,
                "error": str(e).split("\n")[-1][:240],
            })
    out = pd.DataFrame(rows)
    display(out)
    return out


def choose_factor_source(cfg: Config = CFG) -> FactorSource:
    if not cfg.auto_detect_factor_source:
        src = cfg.factor_sources[0]
        progress(f"使用默认因子来源：{src.table}.({_short_expr(src.expr)})", cfg)
        return src

    progress("开始轻量探测 qfa_ocftosales 可用字段", cfg)
    errors: List[str] = []
    for src in cfg.factor_sources:
        expr = qualify_expr(src.expr, "f")
        sql = f"""
        SELECT f.date, f.instrument, ({expr}) AS factor_raw
        FROM {src.table} AS f
        WHERE f.date >= DATE '{cfg.start_date}'
          AND f.date <= DATE '{cfg.end_date}'
          AND ({expr}) IS NOT NULL
          AND ABS(({expr})) < {cfg.factor_abs_cap}
        LIMIT 5
        """
        try:
            tmp = query_df(sql)
            if tmp is not None and not tmp.empty:
                progress(f"选定因子来源：{src.table}.({_short_expr(src.expr)})（{src.desc}）", cfg)
                return src
            errors.append(f"{src.table}.{_short_expr(src.expr)}: 无非空样本")
        except Exception as e:
            errors.append(f"{src.table}.{_short_expr(src.expr)}: {str(e).split(chr(10))[-1][:160]}")
    raise RuntimeError("qfa_ocftosales 候选字段/表达式均不可用：\n" + "\n".join(errors))


# =========================
# 日期映射
# =========================

def get_trading_dates(cfg: Config = CFG) -> pd.DatetimeIndex:
    fetch_end = (pd.Timestamp(cfg.end_date) + pd.Timedelta(days=max(90, cfg.forward_days * 3))).strftime("%Y-%m-%d")
    sql = f"""
    SELECT DISTINCT date
    FROM cn_stock_prefactors
    WHERE date >= DATE '{cfg.start_date}'
      AND date <= DATE '{fetch_end}'
    ORDER BY date
    """
    df = query_df(sql)
    if df.empty:
        raise ValueError("未读取到交易日，请检查日期区间或 cn_stock_prefactors 权限。")
    return pd.DatetimeIndex(pd.to_datetime(df["date"]).drop_duplicates().sort_values())


def build_signal_date_map(cfg: Config, trade_dates: pd.DatetimeIndex) -> pd.DataFrame:
    end_ts = pd.Timestamp(cfg.end_date)
    all_signal_dates = list(trade_dates[trade_dates <= end_ts][:: cfg.rebalance_freq])
    pos = pd.Series(np.arange(len(trade_dates)), index=trade_dates)
    rows = []
    for dt in all_signal_dates:
        i = int(pos.loc[dt])
        next_i = i + 1
        fwd_i = i + cfg.forward_days
        if next_i >= len(trade_dates) or fwd_i >= len(trade_dates):
            continue
        rows.append({
            "signal_date": pd.Timestamp(dt).strftime("%Y-%m-%d"),
            "next_date": pd.Timestamp(trade_dates[next_i]).strftime("%Y-%m-%d"),
            "fwd_date": pd.Timestamp(trade_dates[fwd_i]).strftime("%Y-%m-%d"),
        })
    out = pd.DataFrame(rows)
    if cfg.max_signal_dates is not None:
        out = out.head(int(cfg.max_signal_dates)).copy()
    if out.empty:
        raise ValueError("没有形成可用的 signal_date / next_date / fwd_date 映射。")
    return out


def split_chunks(df: pd.DataFrame, chunk_size: int) -> List[pd.DataFrame]:
    chunk_size = max(1, int(chunk_size))
    return [df.iloc[i:i + chunk_size].copy() for i in range(0, len(df), chunk_size)]


# =========================
# 批量数据读取
# =========================

def load_signal_chunk(date_map_chunk: pd.DataFrame, source: FactorSource, cfg: Config) -> pd.DataFrame:
    signal_dates = date_map_chunk["signal_date"].tolist()
    signal_sql = date_in_sql(signal_dates)

    if source.table == "cn_stock_prefactors":
        expr = qualify_expr(source.expr, "p")
        sql = f"""
        SELECT
            p.date AS signal_date,
            p.instrument,
            p.close AS close_signal,
            p.close_000300SH AS hs300_close_signal,
            p.float_market_cap,
            ({expr}) AS factor_raw
        FROM cn_stock_prefactors AS p
        WHERE p.date IN ({signal_sql})
          AND COALESCE(p.list_sector, 0) != 4
          AND p.is_risk_warning = 0
          AND p.suspended = 0
          AND p.close > 0
          AND p.close_000300SH > 0
          AND p.float_market_cap > 0
          AND ({expr}) IS NOT NULL
          AND ABS(({expr})) < {cfg.factor_abs_cap}
        ORDER BY p.date, p.instrument
        """
    else:
        expr = qualify_expr(source.expr, "f")
        sql = f"""
        SELECT
            p.date AS signal_date,
            p.instrument,
            p.close AS close_signal,
            p.close_000300SH AS hs300_close_signal,
            p.float_market_cap,
            ({expr}) AS factor_raw
        FROM cn_stock_prefactors AS p
        JOIN {source.table} AS f
          ON f.date = p.date AND f.instrument = p.instrument
        WHERE p.date IN ({signal_sql})
          AND COALESCE(p.list_sector, 0) != 4
          AND p.is_risk_warning = 0
          AND p.suspended = 0
          AND p.close > 0
          AND p.close_000300SH > 0
          AND p.float_market_cap > 0
          AND ({expr}) IS NOT NULL
          AND ABS(({expr})) < {cfg.factor_abs_cap}
        ORDER BY p.date, p.instrument
        """
    df = query_df(sql)
    if df.empty:
        return df
    df["signal_date"] = pd.to_datetime(df["signal_date"]).dt.strftime("%Y-%m-%d")
    for c in ["close_signal", "hs300_close_signal", "float_market_cap", "factor_raw"]:
        df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    df = df.dropna(subset=["signal_date", "instrument", "close_signal", "hs300_close_signal", "float_market_cap", "factor_raw"])
    df = df[(df["close_signal"] > 0) & (df["hs300_close_signal"] > 0) & (df["float_market_cap"] > 0)]
    return df.reset_index(drop=True)


def load_next_suspended_chunk(date_map_chunk: pd.DataFrame) -> pd.DataFrame:
    next_dates = date_map_chunk["next_date"].drop_duplicates().tolist()
    next_sql = date_in_sql(next_dates)
    sql = f"""
    SELECT date AS next_date, instrument, suspended AS next_suspended
    FROM cn_stock_prefactors
    WHERE date IN ({next_sql})
    ORDER BY date, instrument
    """
    df = query_df(sql)
    if df.empty:
        return df
    df["next_date"] = pd.to_datetime(df["next_date"]).dt.strftime("%Y-%m-%d")
    df["next_suspended"] = pd.to_numeric(df["next_suspended"], errors="coerce").fillna(1).astype("int8")
    df = df.drop_duplicates(["next_date", "instrument"], keep="last")
    return df.reset_index(drop=True)


def load_forward_price_chunk(date_map_chunk: pd.DataFrame) -> pd.DataFrame:
    fwd_dates = date_map_chunk["fwd_date"].drop_duplicates().tolist()
    fwd_sql = date_in_sql(fwd_dates)
    sql = f"""
    SELECT
        date AS fwd_date,
        instrument,
        close AS close_fwd,
        close_000300SH AS hs300_close_fwd
    FROM cn_stock_prefactors
    WHERE date IN ({fwd_sql})
      AND close > 0
      AND close_000300SH > 0
    ORDER BY date, instrument
    """
    df = query_df(sql)
    if df.empty:
        return df
    df["fwd_date"] = pd.to_datetime(df["fwd_date"]).dt.strftime("%Y-%m-%d")
    for c in ["close_fwd", "hs300_close_fwd"]:
        df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    df = df.dropna(subset=["fwd_date", "instrument", "close_fwd", "hs300_close_fwd"])
    df = df[(df["close_fwd"] > 0) & (df["hs300_close_fwd"] > 0)]
    df = df.drop_duplicates(["fwd_date", "instrument"], keep="last")
    return df.reset_index(drop=True)


def build_panel_chunk(date_map_chunk: pd.DataFrame, source: FactorSource, cfg: Config) -> pd.DataFrame:
    sig = load_signal_chunk(date_map_chunk, source, cfg)
    if sig.empty:
        return pd.DataFrame()
    nxt = load_next_suspended_chunk(date_map_chunk)
    fwd = load_forward_price_chunk(date_map_chunk)
    if nxt.empty or fwd.empty:
        return pd.DataFrame()

    mp = date_map_chunk.copy()
    for c in ["signal_date", "next_date", "fwd_date"]:
        mp[c] = pd.to_datetime(mp[c]).dt.strftime("%Y-%m-%d")

    df = sig.merge(mp, on="signal_date", how="inner")
    del sig
    gc.collect()

    df = df.merge(nxt, on=["next_date", "instrument"], how="left")
    del nxt
    gc.collect()

    df["next_suspended"] = pd.to_numeric(df["next_suspended"], errors="coerce").fillna(1).astype("int8")
    df = df[df["next_suspended"] == 0].copy()
    if df.empty:
        return df

    df = df.merge(fwd, on=["fwd_date", "instrument"], how="inner")
    del fwd
    gc.collect()

    if df.empty:
        return df

    df["ret_fwd"] = df["close_fwd"] / df["close_signal"] - 1.0
    # 同一 fwd_date/signal_date 下沪深300收益对所有股票相同；直接逐行计算更简单，内存影响可控。
    df["bench_ret_fwd"] = df["hs300_close_fwd"] / df["hs300_close_signal"] - 1.0
    df["excess_ret_fwd"] = df["ret_fwd"] - df["bench_ret_fwd"]
    df["date"] = pd.to_datetime(df["signal_date"])

    keep_cols = ["date", "instrument", "float_market_cap", "factor_raw", "ret_fwd", "excess_ret_fwd"]
    df = df[keep_cols].dropna()
    for c in ["float_market_cap", "factor_raw", "ret_fwd", "excess_ret_fwd"]:
        df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    df = df[np.isfinite(df["factor_raw"]) & np.isfinite(df["ret_fwd"]) & np.isfinite(df["excess_ret_fwd"])]
    return df.reset_index(drop=True)


# =========================
# 指标计算
# =========================

def robust_zscore_np(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float64, copy=False)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out
    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        lo, hi = med - 5.0 * mad, med + 5.0 * mad
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])
    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def corr_np(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3:
        return np.nan
    xv = x[valid].astype(np.float64, copy=False)
    yv = y[valid].astype(np.float64, copy=False)
    xv = xv - xv.mean()
    yv = yv - yv.mean()
    denom = np.sqrt(np.dot(xv, xv) * np.dot(yv, yv))
    if not np.isfinite(denom) or denom <= 1e-18:
        return np.nan
    return float(np.dot(xv, yv) / denom)


def rank_np(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method="average").to_numpy(dtype=np.float64, copy=False)


def wls_factor_return_closed_form(y: np.ndarray, f: np.ndarray, mv: np.ndarray) -> Tuple[float, float]:
    y = y.astype(np.float64, copy=False)
    f = f.astype(np.float64, copy=False)
    mv = mv.astype(np.float64, copy=False)
    w = np.sqrt(np.clip(mv, 1.0, None))
    valid = np.isfinite(y) & np.isfinite(f) & np.isfinite(w) & (w > 0)
    n = int(valid.sum())
    if n < 30:
        return np.nan, np.nan
    yv = y[valid]
    fv = f[valid]
    wv = w[valid]
    sw = wv.sum()
    if not np.isfinite(sw) or sw <= 0:
        return np.nan, np.nan
    f_bar = np.sum(wv * fv) / sw
    y_bar = np.sum(wv * yv) / sw
    fc = fv - f_bar
    yc = yv - y_bar
    sxx = np.sum(wv * fc * fc)
    if not np.isfinite(sxx) or sxx <= 1e-18:
        return np.nan, np.nan
    beta = np.sum(wv * fc * yc) / sxx
    alpha = y_bar - beta * f_bar
    resid = yv - alpha - beta * fv
    dof = max(n - 2, 1)
    sigma2 = np.sum(wv * resid * resid) / dof
    se_beta = np.sqrt(max(sigma2 / sxx, 0.0))
    t_value = float(beta / se_beta) if se_beta > 1e-18 else np.nan
    return float(beta), t_value


def calc_cross_section_metrics(g: pd.DataFrame, cfg: Config) -> Optional[Dict[str, float]]:
    if len(g) < cfg.min_cross_section_size:
        return None
    factor_z = robust_zscore_np(g["factor_raw"].to_numpy(dtype=np.float64, copy=False))
    ret = g["ret_fwd"].to_numpy(dtype=np.float64, copy=False)
    excess_ret = g["excess_ret_fwd"].to_numpy(dtype=np.float64, copy=False)
    mv = g["float_market_cap"].to_numpy(dtype=np.float64, copy=False)
    valid = np.isfinite(factor_z) & np.isfinite(ret)
    if valid.sum() < cfg.min_cross_section_size:
        return None
    ic = corr_np(factor_z[valid], ret[valid])
    rank_ic = corr_np(rank_np(factor_z[valid]), rank_np(ret[valid]))
    factor_ret, t_value = wls_factor_return_closed_form(excess_ret, factor_z, mv)
    return {
        "date": g["date"].iloc[0],
        "样本数": int(valid.sum()),
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": factor_ret,
        "t值": t_value,
    }


def calc_metrics_batch(cfg: Config = CFG) -> pd.DataFrame:
    source = choose_factor_source(cfg)
    trade_dates = get_trading_dates(cfg)
    date_map = build_signal_date_map(cfg, trade_dates)
    chunks = split_chunks(date_map, cfg.chunk_signal_dates)
    progress(
        f"交易日数量：{len(trade_dates):,}；信号截面数量：{len(date_map):,}；"
        f"chunk={cfg.chunk_signal_dates}，共 {len(chunks)} 个 chunk",
        cfg,
    )

    rows: List[Dict[str, float]] = []
    for ci, mp in enumerate(chunks, 1):
        s0 = mp["signal_date"].iloc[0]
        s1 = mp["signal_date"].iloc[-1]
        progress(f"处理 chunk {ci}/{len(chunks)}：{s0} ~ {s1}，{len(mp)} 个截面", cfg)
        panel = build_panel_chunk(mp, source, cfg)
        if panel.empty:
            progress("该 chunk 无有效数据，跳过", cfg)
            continue
        m = mem_mb()
        if np.isfinite(m):
            progress(f"chunk 面板 {len(panel):,} 行；内存约 {m:.1f} MB", cfg)

        for dt, g in panel.groupby("date", sort=True, observed=True):
            row = calc_cross_section_metrics(g, cfg)
            if row is not None:
                rows.append(row)

        if cfg.save_metrics_csv and rows:
            pd.DataFrame(rows).sort_values("date").to_csv(cfg.metrics_csv_path, index=False, encoding="utf-8-sig")

        del panel
        gc.collect()

    if not rows:
        raise ValueError("没有足够截面可计算指标，请检查日期区间、字段权限、股票池过滤条件。")
    metrics = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    progress(f"指标计算完成：{len(metrics):,} 个截面", cfg)
    return metrics


def safe_ir(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna()
    std = s.std(ddof=1)
    if len(s) < 2 or not np.isfinite(std) or std <= 1e-18:
        return np.nan
    return float(s.mean() / std)


def make_summary(metrics: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    ic = pd.to_numeric(metrics["IC"], errors="coerce")
    rank_ic = pd.to_numeric(metrics["RankIC"], errors="coerce")
    factor_ret = pd.to_numeric(metrics["因子收益率"], errors="coerce")
    t_value = pd.to_numeric(metrics["t值"], errors="coerce")
    return pd.DataFrame([{
        "因子": cfg.factor_name,
        "起始日": metrics["date"].min().strftime("%Y-%m-%d"),
        "结束日": metrics["date"].max().strftime("%Y-%m-%d"),
        "截面数": int(metrics["date"].nunique()),
        "平均截面样本数": metrics["样本数"].mean(),
        "IC均值": ic.mean(),
        "IC标准差": ic.std(ddof=1),
        "ICIR": safe_ir(ic),
        "IC胜率": (ic > 0).mean(),
        "RankIC均值": rank_ic.mean(),
        "RankIC标准差": rank_ic.std(ddof=1),
        "RankICIR": safe_ir(rank_ic),
        "RankIC胜率": (rank_ic > 0).mean(),
        "因子收益率均值": factor_ret.mean(),
        "因子收益率标准差": factor_ret.std(ddof=1),
        "t值均值": t_value.mean(),
        "|t|均值": t_value.abs().mean(),
        "|t|>2占比": (t_value.abs() > 2).mean(),
        "t均值/t标准差": safe_ir(t_value),
    }])


def format_summary(summary: pd.DataFrame) -> pd.DataFrame:
    out = summary.copy()
    for col in ["截面数"]:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{int(x)}")
    decimal_cols = [
        "平均截面样本数", "IC均值", "IC标准差", "ICIR", "RankIC均值", "RankIC标准差",
        "RankICIR", "因子收益率均值", "因子收益率标准差", "t值均值", "|t|均值", "t均值/t标准差",
    ]
    pct_cols = ["IC胜率", "RankIC胜率", "|t|>2占比"]
    for col in decimal_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
    for col in pct_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.2%}")
    return out


def plot_metrics(metrics: pd.DataFrame, cfg: Config) -> None:
    if not cfg.make_plots or plt is None:
        return
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"], label="IC", linewidth=1.4)
    ax.plot(metrics["date"], metrics["RankIC"], label="RankIC", linewidth=1.4)
    ax.axhline(0, linewidth=1.0, linestyle="--")
    ax.set_title(f"{cfg.factor_name}: IC and RankIC")
    ax.set_xlabel("date")
    ax.set_ylabel("corr")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"].fillna(0).cumsum(), label="IC cumsum", linewidth=1.4)
    ax.plot(metrics["date"], metrics["RankIC"].fillna(0).cumsum(), label="RankIC cumsum", linewidth=1.4)
    ax.axhline(0, linewidth=1.0, linestyle="--")
    ax.set_title(f"{cfg.factor_name}: cumulative IC and RankIC")
    ax.set_xlabel("date")
    ax.set_ylabel("cum corr")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def main() -> Tuple[pd.DataFrame, pd.DataFrame]:
    metrics = calc_metrics_batch(CFG)
    summary = make_summary(metrics, CFG)
    display(format_summary(summary))
    plot_metrics(metrics, CFG)
    return summary, metrics


summary, metrics = main()


[00:00] 使用默认因子来源：cn_stock_prefactors.(net_cffoa_mrq / NULLIF(operating_revenue_mrq, 0))
[00:17] 交易日数量：849；信号截面数量：28；chunk=8，共 4 个 chunk
[00:17] 处理 chunk 1/4：2023-01-03 ~ 2023-11-16，8 个截面


PermissionException: Permission Error: 请在查询表 cn_stock_index_bar1d 时使用 filters 参数指定分区范围（一般为 date 或 instrument ）：dai.query(sql, filters={"date": ["2020-01-01", "2020-02-01"]})。若确需全表扫描，请设置 full_db_scan 参数：dai.query(sql, full_db_scan=True)

该因子在计算查询数据上遇到了问题，于是我们暂时放弃对这个因子的复现